# Steam 할인 패턴 분석

- 이벤트 기준 132건
- 게임 기준 46개 함께 확인
- 반응률과 유지율은 Winsorize 적용
- 시즌 비교는 참고 비교로 해석


## Part 0 환경 설정

윈도우와 맥 모두 한글이 안 깨지게 폰트를 먼저 설정


In [7]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager

warnings.filterwarnings('ignore')


def root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent]:
        if (candidate / 'data').exists() and (candidate / 'figures').exists():
            return candidate
    return cwd


PROJECT_ROOT = root()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURE_DIR = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)


def kfont():
    candidate_paths = [
        PROJECT_ROOT / 'fonts' / 'NanumGothic.ttc',
        PROJECT_ROOT / 'fonts' / 'NanumGothic.ttf',
    ]
    for font_path in candidate_paths:
        if font_path.exists():
            try:
                font_manager.fontManager.addfont(str(font_path))
                return font_manager.FontProperties(fname=str(font_path)).get_name()
            except Exception:
                pass

    installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR', 'Noto Sans KR']:
        if font_name in installed_fonts:
            return font_name
    return 'DejaVu Sans'


FONT_NAME = kfont()
plt.rcParams['font.family'] = FONT_NAME
plt.rcParams['font.sans-serif'] = [FONT_NAME, 'Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

FIG_W, FIG_H = 10, 6
DPI = 300

print('한글 폰트', FONT_NAME)
print('프로젝트 루트', PROJECT_ROOT)
print('저장 폴더', FIGURE_DIR)


한글 폰트 Nanum Gothic
프로젝트 루트 C:\Users\alvin\OneDrive\바탕 화면\WorkSpace\steam_discount_analysis
저장 폴더 C:\Users\alvin\OneDrive\바탕 화면\WorkSpace\steam_discount_analysis\figures


## Part 1 데이터 준비

- 분석 테이블 불러오기
- Winsorize 적용
- 게임 단위 요약 테이블 생성

해석할 때는 이벤트 단위와 게임 단위를 같이 봄


In [8]:
analysis_df = pd.read_csv(DATA_DIR / 'analysis_df.csv')
discount_history = pd.read_csv(DATA_DIR / 'discount_history.csv')
review_daily = pd.read_csv(DATA_DIR / 'review_daily.csv')

analysis_df['discount_start'] = pd.to_datetime(analysis_df['discount_start'])
analysis_df['discount_end'] = pd.to_datetime(analysis_df['discount_end'])
discount_history['discount_start'] = pd.to_datetime(discount_history['discount_start'])
discount_history['discount_end'] = pd.to_datetime(discount_history['discount_end'])
review_daily['date'] = pd.to_datetime(review_daily['date'])


def clip95(s, lower_pct=5, upper_pct=95):
    lo = np.percentile(s, lower_pct)
    hi = np.percentile(s, upper_pct)
    return s.clip(lo, hi), lo, hi


analysis_df['reaction_rate_raw'] = analysis_df['reaction_rate']
analysis_df['sustained_rate_raw'] = analysis_df['sustained_rate']
analysis_df['reaction_rate'], rr_lo, rr_hi = clip95(analysis_df['reaction_rate'])
analysis_df['sustained_rate'], sr_lo, sr_hi = clip95(analysis_df['sustained_rate'])

game_df = (
    analysis_df
    .groupby(['appid', 'name', 'genre_category'], as_index=False)
    .agg(
        n_events=('appid', 'count'),
        reaction_median=('reaction_rate', 'median'),
        sustained_median=('sustained_rate', 'median'),
        discount_mean=('discount_pct', 'mean'),
        has_seasonal=('is_seasonal_sale', 'max'),
    )
)

GENRE_ORDER = ['RPG', 'Adventure', 'Strategy/Simulation', 'Casual/Lightweight', 'Action']
GENRE_COLORS = {
    'RPG': '#4C72B0',
    'Adventure': '#DD8452',
    'Strategy/Simulation': '#55A868',
    'Casual/Lightweight': '#C44E52',
    'Action': '#8172B2',
}

print('이벤트 수', len(analysis_df))
print('게임 수', game_df['appid'].nunique())
print('Winsorize 경계', rr_lo, rr_hi, sr_lo, sr_hi)
print('장르 분포:')
print(analysis_df.groupby('genre_category')['appid'].count())

이벤트 수 263
게임 수 53
Winsorize 경계 -0.29468999999999995 2.022379999999993 -0.43026999999999993 1.0654799999999993
장르 분포:
genre_category
Action                 32
Adventure              87
Casual/Lightweight     57
RPG                    33
Strategy/Simulation    54
Name: appid, dtype: int64


## Part 2 장르별 반응

이벤트 단위 중앙값을 기준으로 장르별 반응/유지 패턴을 비교

### 보는 법
- 파란 막대: 반응률 중앙값
- 주황 막대: 유지율 중앙값
- x축 `n=`은 장르별 이벤트 수
- 막대가 높을수록 해당 지표가 큰 편
- Action은 표본(이벤트 9건)이 작아 탐색적으로 해석


In [9]:
# PPT 표 기준과 동일한 이벤트 단위 집계
chart1_table = (
    analysis_df.groupby('genre_category', as_index=False)
    .agg(
        reaction_med=('reaction_rate', 'median'),
        sustained_med=('sustained_rate', 'median'),
        n_events=('appid', 'count'),
    )
    .sort_values('reaction_med', ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(chart1_table))
width = 0.36

bars1 = ax.bar(
    x - width / 2,
    chart1_table['reaction_med'],
    width,
    label='반응률 중앙값',
    color='#4C72B0',
    edgecolor='black',
    alpha=0.85,
)
bars2 = ax.bar(
    x + width / 2,
    chart1_table['sustained_med'],
    width,
    label='유지율 중앙값',
    color='#DD8452',
    edgecolor='black',
    alpha=0.85,
)

ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.grid(axis='y', alpha=0.3)
ax.set_ylabel('중앙값')
ax.set_title('장르별 반응/유지 (이벤트 기준)')

xticklabels = [f"{g}\n(n={int(n)})" for g, n in zip(chart1_table['genre_category'], chart1_table['n_events'])]
ax.set_xticks(x)
ax.set_xticklabels(xticklabels)

for bars in (bars1, bars2):
    for bar in bars:
        h = bar.get_height()
        va = 'bottom' if h >= 0 else 'top'
        off = 0.012 if h >= 0 else -0.012
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + off,
            f'{h:+.3f}',
            ha='center',
            va=va,
            fontsize=9,
        )

if 'Action' in chart1_table['genre_category'].values:
    action_i = int(chart1_table.index[chart1_table['genre_category'] == 'Action'][0])
    action_y = float(chart1_table.loc[action_i, 'reaction_med'])
    ax.annotate(
        'Action: 이벤트 9건\n탐색적 해석',
        xy=(x[action_i] - width / 2, action_y),
        xytext=(x[action_i] - 0.85, action_y + 0.11),
        fontsize=8,
        color='dimgray',
        arrowprops=dict(arrowstyle='->', color='dimgray', lw=0.8),
        bbox=dict(boxstyle='round,pad=0.25', facecolor='lightyellow', edgecolor='gray', alpha=0.9),
    )

ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart1_genre.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart1_genre.png')

print('=== 장르별 핵심 수치 (이벤트 기준) ===')
print(
    chart1_table.rename(
        columns={
            'genre_category': '장르',
            'reaction_med': '반응률 중앙값',
            'sustained_med': '유지율 중앙값',
            'n_events': '이벤트 수',
        }
    )
    .assign(**{'반응률 중앙값': lambda d: d['반응률 중앙값'].round(3), '유지율 중앙값': lambda d: d['유지율 중앙값'].round(3)})
    .to_string(index=False)
)


저장 완료 chart1_genre.png
=== 장르별 핵심 수치 (이벤트 기준) ===
                 장르  반응률 중앙값  유지율 중앙값  이벤트 수
          Adventure    0.280    0.135     87
             Action    0.269    0.100     32
 Casual/Lightweight    0.256    0.150     57
Strategy/Simulation    0.205    0.059     54
                RPG    0.200    0.057     33


In [10]:
# ── 장르별 반응률 수치 표 ──────────────────────────────────────────
# 이벤트 단위 집계, Winsorize 적용 값 기준
# PPT 결론 슬라이드 수치 출처

genre_table = (
    analysis_df.groupby('genre_category')
    .agg(
        게임수=('appid', 'nunique'),
        이벤트수=('appid', 'count'),
        반응률_중앙값=('reaction_rate', 'median'),
        유지율_중앙값=('sustained_rate', 'median'),
        할인율_평균=('discount_pct', 'mean'),
    )
    .reindex(GENRE_ORDER)
    .round(3)
)
genre_table.index.name = '장르'
genre_table.columns = ['게임 수', '이벤트 수', '반응률 중앙값', '유지율 중앙값', '평균 할인율(%)']

print("=== 장르별 반응률 수치 표 (Winsorize 5~95%, 이벤트 기준) ===")
print(genre_table.to_string())
print()
print("* Action: 이벤트 수 9건으로 타 장르 대비 표본 소수 → 발표 시 탐색적 해석 명시 필요")
genre_table


=== 장르별 반응률 수치 표 (Winsorize 5~95%, 이벤트 기준) ===
                     게임 수  이벤트 수  반응률 중앙값  유지율 중앙값  평균 할인율(%)
장르                                                           
RPG                    10     33    0.200    0.057     61.000
Adventure              11     87    0.280    0.135     62.161
Strategy/Simulation    11     54    0.205    0.059     52.167
Casual/Lightweight     11     57    0.256    0.150     42.491
Action                 10     32    0.269    0.100     60.156

* Action: 이벤트 수 9건으로 타 장르 대비 표본 소수 → 발표 시 탐색적 해석 명시 필요


,게임 수,이벤트 수,반응률 중앙값,유지율 중앙값,평균 할인율(%)
장르,,,,,
RPG,10,33,0.200,0.057,61.000
Adventure,11,87,0.280,0.135,62.161
Strategy/Simulation,11,54,0.205,0.059,52.167
Casual/Lightweight,11,57,0.256,0.150,42.491
Action,10,32,0.269,0.100,60.156


## Part 3 할인율과 반응

게임별 평균 할인율과 반응률 중앙값을 산점도로 비교

### 보는 법
- 점 하나가 게임 하나를 뜻함
- 오른쪽으로 갈수록 평균 할인율이 큼
- 위로 갈수록 반응률 중앙값이 큼
- 점들이 오른쪽 위 방향으로 모이면 할인율과 반응이 함께 커지는 경향이 있다는 뜻
- 상관계수는 전체 흐름을 참고하는 숫자일 뿐 원인 관계를 뜻하지 않음


In [11]:
# Q1 chart: event-level discount vs reaction
# Keep Korean labels via unicode escapes to avoid shell encoding issues
if 'FONT_NAME' not in globals():
    try:
        FONT_NAME = kfont() if 'kfont' in globals() else 'DejaVu Sans'
    except Exception:
        FONT_NAME = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_NAME
plt.rcParams['font.sans-serif'] = [FONT_NAME, 'Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(10, 6))
for genre in GENRE_ORDER:
    sub = analysis_df[analysis_df['genre_category'] == genre]
    ax.scatter(
        sub['discount_pct'],
        sub['reaction_rate'],
        color=GENRE_COLORS[genre],
        label=genre,
        alpha=0.72,
        s=44,
    )

rho, pval = spearmanr(analysis_df['discount_pct'], analysis_df['reaction_rate'])
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('\ud3c9\uade0 \ud560\uc778\uc728(%)')
ax.set_ylabel('\ubc18\uc751\ub960')
ax.set_title('\ud560\uc778\uc728\uacfc \ubc18\uc751 (\uc774\ubca4\ud2b8 \uae30\uc900)')
ax.legend(fontsize=9, loc='center right')
ax.text(
    0.02,
    0.97,
    f'\uc774\ubca4\ud2b8 \uae30\uc900 (n={len(analysis_df)})\nSpearman \u03c1 = {rho:.3f}\np = {pval:.3f}',
    transform=ax.transAxes,
    va='top',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85),
)
ax.text(
    0.98,
    0.03,
    '\ucc38\uace0: \ud560\uc778\uc728 \uac12\uc740 \uc2e4\uc81c \ub370\uc774\ud130\uc5d0\uc11c \uc8fc\ub85c 5% \ub2e8\uc704',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=8,
    color='gray',
)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart2_discount.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('\uc800\uc7a5 \uc644\ub8cc chart2_discount.png')


저장 완료 chart2_discount.png


## Part 4 반응과 유지

게임별 반응과 유지가 같이 움직이는지 확인

### 보는 법
- 점 하나가 게임 하나를 뜻함
- 오른쪽으로 갈수록 할인 중 반응이 큼
- 위로 갈수록 할인 후 유지가 큼
- 오른쪽 위는 반응도 크고 유지도 큰 경우
- 오른쪽 아래는 할인 중 반응은 컸지만 할인 후 유지가 약한 경우


In [12]:
# PPT table-aligned labels for chart3
genre_profile = (
    analysis_df.groupby('genre_category', as_index=False)
    .agg(
        reaction_med=('reaction_rate', 'median'),
        sustained_med=('sustained_rate', 'median'),
        n_events=('appid', 'count'),
    )
    .set_index('genre_category')
    .reindex(GENRE_ORDER)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))

label_offsets = {
    'RPG': (0.010, -0.005),
    'Adventure': (0.012, 0.012),
    'Strategy/Simulation': (0.010, 0.014),
    'Casual/Lightweight': (0.012, 0.010),
    'Action': (0.012, 0.012),
}

for _, row in genre_profile.iterrows():
    g = row['genre_category']
    r = row['reaction_med']
    s = row['sustained_med']
    n = int(row['n_events'])
    dx, dy = label_offsets.get(g, (0.012, 0.012))

    ax.scatter(
        r, s,
        color=GENRE_COLORS[g],
        s=170,
        edgecolor='black',
        linewidth=1.0,
        zorder=3,
        label=g,
    )
    ax.plot([r, r], [0, s], color=GENRE_COLORS[g], alpha=0.25, linewidth=1.2, zorder=1)
    ax.annotate(
        f"{g}\n{r:+.3f} / {s:+.3f}\n(n={n})",
        xy=(r, s),
        xytext=(r + dx, s + dy),
        textcoords='data',
        fontsize=8.5,
        ha='left',
        va='bottom',
        zorder=5,
        bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.65),
    )

mx = max(genre_profile['reaction_med'].max(), genre_profile['sustained_med'].max()) + 0.05
mn = min(genre_profile['reaction_med'].min(), genre_profile['sustained_med'].min(), 0) - 0.03
ax.plot([mn, mx], [mn, mx], color='gray', linestyle='--', linewidth=1, alpha=0.8)
ax.text(0.02, 0.05, 'y=x (반응=유지)', transform=ax.transAxes, fontsize=8, color='gray')

ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlim(mn, mx)
ax.set_ylim(mn, mx)
ax.set_xlabel('Engagement 반응률 중앙값')
ax.set_ylabel('Engagement 유지율 중앙값')
ax.set_title('즉시 반응 vs 유지 (장르 중앙값, 이벤트 기준)')
ax.grid(alpha=0.25)

handles, labels = ax.get_legend_handles_labels()
uniq = dict(zip(labels, handles))
ax.legend(uniq.values(), uniq.keys(), fontsize=9, loc='upper left', title='장르')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart3_keep.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart3_keep.png')

저장 완료 chart3_keep.png


In [13]:
# ── Part 7  할인 빈도와 반응 (연구 질문 4) ──────────────────────────
# 보는 법 (왼쪽): 게임별 이벤트 수 중앙값 기준으로 저빈도/고빈도 두 그룹 나눠 비교
#               막대가 높을수록 그 그룹의 반응률 중앙값이 큼
# 보는 법 (오른쪽): 게임별 이벤트 수 vs 반응률 산점도
#                  상관계수는 전체 흐름 참고용이며 인과관계를 의미하지 않음
# 자주 할인하는 게임과 가끔 하는 게임의 반응 패턴 차이 정량 확인

freq_df = (
    analysis_df.groupby(['appid', 'name', 'genre_category'], as_index=False)
    .agg(n_events=('appid', 'count'), reaction_median=('reaction_rate', 'median'))
)
cut = int(freq_df['n_events'].median())
freq_df['freq_group'] = freq_df['n_events'].apply(
    lambda x: f'고빈도 ({cut + 1}건+)' if x > cut else f'저빈도 ({cut}건 이하)'
)
freq_summary = freq_df.groupby('freq_group')['reaction_median'].agg(['median', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: 저빈도/고빈도 막대
bars_f = axes[0].bar(
    freq_summary.index, freq_summary['median'],
    color=['#5B9BD5', '#ED7D31'], edgecolor='black', alpha=0.8, width=0.4
)
axes[0].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[0].set_title('할인 빈도별 반응률 (Q4)')
axes[0].set_ylabel('반응률 중앙값')
axes[0].grid(axis='y', alpha=0.3)
for bar, idx in zip(bars_f, freq_summary.index):
    med = freq_summary.loc[idx, 'median']
    n   = int(freq_summary.loc[idx, 'count'])
    axes[0].text(bar.get_x() + bar.get_width() / 2, med + 0.015,
                 f'{med:.2f}\nn={n}', ha='center', va='bottom', fontsize=10)
axes[0].text(0.98, 0.03, f'게임 기준 이벤트 수 중앙값({cut}건)으로 구분',
             transform=axes[0].transAxes, ha='right', va='bottom', fontsize=8, color='gray')

# 오른쪽: 이벤트 수 vs 반응률 산점도
for genre in GENRE_ORDER:
    sub = freq_df[freq_df['genre_category'] == genre]
    axes[1].scatter(sub['n_events'], sub['reaction_median'],
                    color=GENRE_COLORS[genre], label=genre, alpha=0.75, s=55)
rho_q4, p_q4 = spearmanr(freq_df['n_events'], freq_df['reaction_median'])
axes[1].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[1].set_xlabel('할인 이벤트 수 (게임별)')
axes[1].set_ylabel('반응률 중앙값')
axes[1].set_title('할인 빈도와 반응 산점도')
axes[1].legend(fontsize=8)
axes[1].text(0.02, 0.97, f'Spearman ρ = {rho_q4:.2f}  p = {p_q4:.3f}',
             transform=axes[1].transAxes, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart6_frequency.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart6_frequency.png')
print()
print(freq_summary.to_string())


저장 완료 chart6_frequency.png

              median  count
freq_group                 
고빈도 (4건+)    0.24475     20
저빈도 (3건 이하)  0.29855     33


## Part 5 시즌 비교

시즌 이벤트가 있는 게임과 없는 게임을 참고 비교

### 보는 법
- 막대 위 숫자는 반응률 중앙값
- `n=`은 게임 수
- 시즌과 비시즌의 높이 차이는 참고 비교로만 봄
- 장르 구성과 게임 구성이 다를 수 있어서 시즌 효과라고 단정하면 안 됨


In [14]:
season_summary = (
    game_df.groupby('has_seasonal')['reaction_median']
    .agg(['median', 'count'])
    .rename(index={False: '비시즌', True: '시즌'})
)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(season_summary.index, season_summary['median'], color=['#87CEEB', '#E9967A'], edgecolor='black', alpha=0.8)
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('구분')
ax.set_ylabel('반응률 중앙값')
ax.set_title('시즌 참고 비교')
ax.grid(axis='y', alpha=0.3)

for bar, idx in zip(bars, season_summary.index):
    med = season_summary.loc[idx, 'median']
    n = int(season_summary.loc[idx, 'count'])
    ax.text(bar.get_x() + bar.get_width() / 2, med + 0.03, f'{med:.2f}\n n={n}', ha='center', va='bottom', fontsize=10)

ax.text(0.98, 0.03, '참고 비교\n장르 구성 차이 있음', transform=ax.transAxes, ha='right', va='bottom', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart4_season.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart4_season.png')


저장 완료 chart4_season.png


## Part 6 사례 추이

설명용 사례 두 개를 골라 실제 리뷰 흐름을 확인

### 보는 법
- 위 그래프는 반응이 크게 나온 이벤트 사례
- 아래 그래프는 반복 할인이 많은 게임 사례
- 선이 위로 튈수록 그 시점의 일별 리뷰 수가 많다는 뜻
- 색칠된 구간은 할인 기간
- 사례는 이해를 돕기 위한 예시이지 전체를 대표한다고 단정하면 안 됨


In [15]:
top_event = analysis_df.sort_values('reaction_rate', ascending=False).iloc[0]

# 사례 2: 반복 할인 게임 — PAYDAY 2 (appid 218620, 분석 대상 중 이벤트 2위 9건)
repeat_game = 218620
repeat_name = analysis_df.loc[analysis_df['appid'] == repeat_game, 'name'].iloc[0]

# Case 1: 반응 최대 이벤트 전후 타이트한 윈도우
case1_start = top_event['discount_start'] - pd.Timedelta(days=30)
case1_end   = top_event['discount_end']   + pd.Timedelta(days=14)
case1_reviews = (
    review_daily[
        (review_daily['appid'] == top_event['appid']) &
        (review_daily['date'] >= case1_start) &
        (review_daily['date'] <= case1_end)
    ].sort_values('date')
)

# Case 2: analysis_df 기준 유효 이벤트 구간만 (리뷰 없는 앞 구간 제거)
repeat_valid = analysis_df[analysis_df['appid'] == repeat_game].sort_values('discount_start')
case2_start = repeat_valid['discount_start'].min() - pd.Timedelta(days=30)
case2_end   = repeat_valid['discount_end'].max()   + pd.Timedelta(days=14)

repeat_reviews = (
    review_daily[
        (review_daily['appid'] == repeat_game) &
        (review_daily['date'] >= case2_start) &
        (review_daily['date'] <= case2_end)
    ].sort_values('date')
)
repeat_events = (
    discount_history[
        (discount_history['appid'] == repeat_game) &
        (discount_history['discount_start'] >= case2_start) &
        (discount_history['discount_end']   <= case2_end)
    ].sort_values('discount_start')
)

fig, axes = plt.subplots(2, 1, figsize=(11, 8))

axes[0].plot(case1_reviews['date'], case1_reviews['daily_reviews'], color='#355C7D', linewidth=1.8)
axes[0].axvspan(top_event['discount_start'], top_event['discount_end'], color='#F08A5D', alpha=0.25)
axes[0].set_title(f'사례 1  {top_event["name"]}')
axes[0].set_ylabel('일별 리뷰 수')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
axes[0].tick_params(axis='x', rotation=20)

axes[1].plot(repeat_reviews['date'], repeat_reviews['daily_reviews'], color='#2A9D8F', linewidth=1.5)
for _, ev in repeat_events.iterrows():
    axes[1].axvspan(ev['discount_start'], ev['discount_end'], color='#E76F51', alpha=0.18)
axes[1].set_title(f'사례 2  {repeat_name}  (유효 할인 이벤트 {len(repeat_valid)}건)')
axes[1].set_ylabel('일별 리뷰 수')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart5_cases.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart5_cases.png')
print(f'사례 1: {top_event["name"]}')
print(f'사례 2: {repeat_name}  구간 {case2_start.date()} ~ {case2_end.date()}  이벤트 {len(repeat_valid)}건')


저장 완료 chart5_cases.png
사례 1: Cities: Skylines
사례 2: PAYDAY 2  구간 2025-05-06 ~ 2026-04-09  이벤트 9건


In [16]:
# ── Part 8  인기도 통제 후 장르별 반응 (통제 변수 확인) ─────────────
# 보는 법:
# - total_reviews(전체 리뷰 수) 중앙값 기준으로 인기도 상/하 두 그룹으로 나눔
# - 같은 장르를 인기도 그룹별로 나란히 비교
# - 두 그룹에서 장르 순서가 비슷하게 유지되면 "인기도 통제 후에도 장르 차이 존재"로 해석 가능
# - 인기도 그룹 구성이 다를 수 있으므로 인과 결론은 아님

pop_median = analysis_df['total_reviews'].median()
pop_label_hi = f'고인기 (리뷰 {pop_median / 10000:.0f}만 이상)'
pop_label_lo = f'저인기 (리뷰 {pop_median / 10000:.0f}만 미만)'
analysis_df['pop_group'] = analysis_df['total_reviews'].apply(
    lambda x: pop_label_hi if x >= pop_median else pop_label_lo
)

pop_genre = (
    analysis_df.groupby(['pop_group', 'genre_category'])['reaction_rate']
    .median()
    .unstack('genre_category')
    .reindex(columns=GENRE_ORDER)
)

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(GENRE_ORDER))
width = 0.35
group_colors = ['#4472C4', '#ED7D31']

for i, (grp, color) in enumerate(zip(pop_genre.index.tolist(), group_colors)):
    vals = pop_genre.loc[grp].values
    b = ax.bar(x + (i - 0.5) * width, vals, width, label=grp, color=color, edgecolor='black', alpha=0.8)
    for bar, val in zip(b, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.015,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(GENRE_ORDER)
ax.set_ylabel('반응률 중앙값')
ax.set_title('인기도 그룹별 장르 반응률 (통제 변수 확인)')
ax.legend(title='인기도 그룹', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.text(0.02, 0.97,
        f'인기도 기준: 전체 리뷰 수 중앙값 {pop_median:,.0f}건\n'
        '두 그룹에서 장르 패턴 유지 여부 확인용',
        transform=ax.transAxes, va='top', fontsize=8, color='gray',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart7_popularity.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart7_popularity.png')
print()
print("=== 인기도 그룹 × 장르별 반응률 중앙값 ===")
print(pop_genre.round(3).to_string())


저장 완료 chart7_popularity.png

=== 인기도 그룹 × 장르별 반응률 중앙값 ===
genre_category     RPG  Adventure  Strategy/Simulation  Casual/Lightweight  Action
pop_group                                                                         
고인기 (리뷰 26만 이상)  0.161      0.200                0.228               0.276   0.269
저인기 (리뷰 26만 미만)  0.265      0.324                0.127               0.146     NaN


In [17]:
# ── 결론 수치 요약 ─────────────────────────────────────────────────
# Q1~Q4 연구 질문에 대한 수치 기반 답변
# 이 셀 출력값을 PPT 결론 슬라이드 수치 근거로 사용

print("=" * 60)
print("핵심 분석 결론 (수치 기반)")
print("=" * 60)

# 장르별 집계
gt = (
    analysis_df.groupby('genre_category')
    .agg(
        n_games=('appid', 'nunique'),
        n_events=('appid', 'count'),
        reaction_med=('reaction_rate', 'median'),
        sustained_med=('sustained_rate', 'median'),
        discount_mean=('discount_pct', 'mean'),
    )
    .reindex(GENRE_ORDER)
)

print("\n[장르별 핵심 수치]")
print(f"{'장르':<25} {'반응률':>8} {'유지율':>8} {'이벤트':>7}  비고")
print("-" * 60)
for g in GENRE_ORDER:
    r = gt.loc[g, 'reaction_med']
    s = gt.loc[g, 'sustained_med']
    n = int(gt.loc[g, 'n_events'])
    note = " ← 이벤트 소수, 탐색적" if n < 15 else ""
    print(f"{g:<25} {r:>+8.3f} {s:>+8.3f} {n:>7}건{note}")

# Q1: 할인율 vs 반응률 상관
rho_disc, p_disc = spearmanr(analysis_df['discount_pct'], analysis_df['reaction_rate'])
print(f"\n[Q1] 할인율 ↔ 반응률 상관")
print(f"     이벤트 기준 Spearman ρ = {rho_disc:.3f}  (p = {p_disc:.3f})")
if abs(rho_disc) < 0.2:
    print("     → 약한 상관. 할인율이 높다고 반응률이 항상 비례하지는 않음")
elif rho_disc > 0:
    print(f"     → 양의 상관 존재 (ρ={rho_disc:.2f}), 단 설명력 제한적")
else:
    print(f"     → 음의 상관 (ρ={rho_disc:.2f}), 고할인이 오히려 약한 경향")

# Q2: 장르별 차이
best_g  = gt['reaction_med'].idxmax()
worst_g = gt['reaction_med'].idxmin()
spread  = gt['reaction_med'].max() - gt['reaction_med'].min()
print(f"\n[Q2] 장르별 반응률 차이")
print(f"     최고: {best_g} ({gt.loc[best_g,'reaction_med']:+.3f})")
print(f"     최저: {worst_g} ({gt.loc[worst_g,'reaction_med']:+.3f})")
print(f"     장르 간 중앙값 격차: {spread:.3f}")

# Q3: 단발성 vs 지속형
print(f"\n[Q3] 반응(즉시) vs 유지(지속) 패턴")
for g in GENRE_ORDER:
    r = gt.loc[g, 'reaction_med']
    s = gt.loc[g, 'sustained_med']
    gap = r - s
    if r > 0.05 and gap > 0.1:
        label = "단발성 (반응>유지)"
    elif s >= r - 0.05 and r > 0:
        label = "지속형 (유지 견조)"
    elif r <= 0:
        label = "반응 미약"
    else:
        label = "혼재"
    print(f"     {g:<25} 반응 {r:+.3f} / 유지 {s:+.3f}  → {label}")

# Q4: 할인 빈도
freq_df2 = (
    analysis_df.groupby('appid')
    .agg(n_ev=('appid', 'count'), r_med=('reaction_rate', 'median'))
)
cut2 = int(freq_df2['n_ev'].median())
hi = freq_df2[freq_df2['n_ev'] > cut2]['r_med'].median()
lo = freq_df2[freq_df2['n_ev'] <= cut2]['r_med'].median()
rho_freq, p_freq = spearmanr(freq_df2['n_ev'], freq_df2['r_med'])
print(f"\n[Q4] 할인 빈도별 반응률")
print(f"     저빈도({cut2}건 이하) 중앙값: {lo:+.3f}  고빈도({cut2+1}건+) 중앙값: {hi:+.3f}")
print(f"     빈도↔반응 Spearman ρ = {rho_freq:.3f}  (p = {p_freq:.3f})")

print(f"\n[데이터 한계]")
print(f"  - Action: 이벤트 {int(gt.loc['Action','n_events'])}건 (타 장르 27~39건 대비 소수) → 탐색적 해석에 한정")
print(f"  - 유효 이벤트 {len(analysis_df)}건 / 수집 이벤트 692건 (리뷰 커버리지 부족 등으로 {692 - len(analysis_df)}건 제외)")
print(f"  - 분석 기준: 리뷰 작성 수 (구매·플레이타임 직접 측정 아님)")
print("=" * 60)


핵심 분석 결론 (수치 기반)

[장르별 핵심 수치]
장르                             반응률      유지율     이벤트  비고
------------------------------------------------------------
RPG                         +0.201   +0.057      33건
Adventure                   +0.280   +0.135      87건
Strategy/Simulation         +0.205   +0.059      54건
Casual/Lightweight          +0.256   +0.150      57건
Action                      +0.269   +0.100      32건

[Q1] 할인율 ↔ 반응률 상관
     이벤트 기준 Spearman ρ = 0.179  (p = 0.004)
     → 약한 상관. 할인율이 높다고 반응률이 항상 비례하지는 않음

[Q2] 장르별 반응률 차이
     최고: Adventure (+0.280)
     최저: RPG (+0.201)
     장르 간 중앙값 격차: 0.079

[Q3] 반응(즉시) vs 유지(지속) 패턴
     RPG                       반응 +0.201 / 유지 +0.057  → 단발성 (반응>유지)
     Adventure                 반응 +0.280 / 유지 +0.135  → 단발성 (반응>유지)
     Strategy/Simulation       반응 +0.205 / 유지 +0.059  → 단발성 (반응>유지)
     Casual/Lightweight        반응 +0.256 / 유지 +0.150  → 단발성 (반응>유지)
     Action                    반응 +0.269 / 유지 +0.100  → 단발성 (반응>유지)

[Q4] 할인 빈도별 반응률
     저빈도(3건